In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI

SEED = 42

# True: rebuild inputs and the selected embeddings, then save those results.
# False: skip that work and load the saved train, validation, and test embedding frames from data/processed/embeddings.
REBUILD = False

# RF configuration; disable class weighting for an unweighted comparison.
feature_cols = [
    "embedding", "condition_vec", "days", "usefulCount_log"
]
rf_params = {
    "numTrees": 200,
    "maxDepth": 16,
    "minInstancesPerNode": 1,
    "seed": 42,
}
balance_classes = True
save_model = True

# `spark.driver.memory` is read only when the JVM starts. Restart the kernel after changing it.
spark = (
    SparkSession.builder 
    .master("local[*]")
    .appName("DrugReviewClassification")
    .config("spark.driver.memory", "4g")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark_context = spark.sparkContext
spark_context.setLogLevel("ERROR")
data_oi = DataOI()


In [ ]:
from src.cleaning import data_cleaning
from src.text_features import text_feature_engineering
from src.encoder import encoder
from src.split import split_and_weight

if REBUILD:
    df_raw = data_oi.load_raw_data()
    df_cleaned = data_cleaning(df_raw)
    df_cleaned.cache().count()
    df_engineered = text_feature_engineering(df_cleaned,True)
    df_engineered.cache().count()
    df_train, df_valid, df_test = split_and_weight(df_engineered, SEED)
    df_train, df_valid, df_test = encoder(df_train, df_valid, df_test)
    data_oi.save_split_data(df_train, df_valid, df_test)


In [ ]:
from src.embeddings import EmbeddingModel

# Toggle embeddings; multiple True entries are allowed. Order = run order.
USE_EMBEDDINGS = {"tfidf": True, "glove": False}
available_embeddings = sorted(
    set(EmbeddingModel.train_registry) & set(EmbeddingModel.predict_registry)
)
unknown = [name for name in USE_EMBEDDINGS if name not in available_embeddings]
if unknown:
    raise KeyError(
        f"USE_EMBEDDINGS names {unknown} not in paired registries {available_embeddings}"
    )
embedding_names = [name for name, on in USE_EMBEDDINGS.items() if on]
if not embedding_names:
    raise ValueError("Enable at least one entry in USE_EMBEDDINGS")

if REBUILD:
    embedding_model = EmbeddingModel(df_train, df_valid, df_test)
    for name in embedding_names:
        print(f"Embedding Model {name} Training")
        EmbeddingModel.train_registry[name](embedding_model)
    train_embedding_dict = {}
    valid_embedding_dict = {}
    test_embedding_dict = {}
    for name in embedding_names:
        print(f"Embedding Model {name} Running")
        train_embedding_dict[name], valid_embedding_dict[name], test_embedding_dict[name] = (
            EmbeddingModel.predict_registry[name](embedding_model)
        )
        data_oi.save_embedding_data(
            name,
            train_embedding_dict[name],
            valid_embedding_dict[name],
            test_embedding_dict[name],
        )
else:
    train_embedding_dict, valid_embedding_dict, test_embedding_dict = data_oi.load_embedding_data()

for split_name, frames in (
    ("train", train_embedding_dict),
    ("valid", valid_embedding_dict),
    ("test", test_embedding_dict),
):
    missing = [name for name in embedding_names if name not in frames]
    if missing:
        raise FileNotFoundError(
            f"USE_EMBEDDINGS selected {missing}, missing from {split_name} cache; "
            f"available: {sorted(frames)}. Run with REBUILD=True to create it."
        )

train_embedding_dict = {name: train_embedding_dict[name] for name in embedding_names}
valid_embedding_dict = {name: valid_embedding_dict[name] for name in embedding_names}
test_embedding_dict = {name: test_embedding_dict[name] for name in embedding_names}
print(f"Embeddings in this run: {embedding_names}")


In [ ]:
from src.analytics import AnalyticsModel

# Toggle classifiers; multiple True entries are allowed. Order = run order.
# Add other registered classifier names here when their modules are available.
USE_MODELS = {"lr_improved": False, "random_forest": True, "svm_l0": False}
unknown = [name for name in USE_MODELS if name not in AnalyticsModel.registry]
if unknown:
    raise KeyError(
        f"USE_MODELS names {unknown} not in registry {sorted(AnalyticsModel.registry)}"
    )
analytics_names = [name for name, on in USE_MODELS.items() if on]
if not analytics_names:
    raise ValueError("Enable at least one entry in USE_MODELS")
print(f"Analytics models in this run: {analytics_names}")

analytics_models = {}
for embedding_name in train_embedding_dict:
    analytics_models[embedding_name] = {
        name: AnalyticsModel(
            embedding_name,
            train_embedding_dict[embedding_name],
            valid_embedding_dict[embedding_name],
            test_embedding_dict[embedding_name],
        )
        for name in analytics_names
    }


In [ ]:
train_pred_dict = {}
valid_pred_dict = {}
test_pred_dict = {}

for embedding_name in train_embedding_dict:
    for analytics_name in analytics_names:
        print(f"Embedding Model {embedding_name} Analytics Model {analytics_name} Running")
        kwargs = {}
        if analytics_name == "random_forest":
            print("Feature columns:", feature_cols)
            print("Random forest parameters:", rf_params)
            print("Balance classes:", balance_classes)
            print("Save model:", save_model)
            kwargs = {
                "feature_cols": feature_cols,
                "rf_params": rf_params,
                "balance_classes": balance_classes,
                "save_model": save_model,
            }
        # Format: analytics__embedding.
        model_name = f"{analytics_name}__{embedding_name}"
        train_pred_dict[model_name], valid_pred_dict[model_name], test_pred_dict[model_name] = (
            AnalyticsModel.registry[analytics_name](
                analytics_models[embedding_name][analytics_name], **kwargs
            )
        )
        data_oi.save_prediction_data(
            model_name,
            train_pred_dict[model_name],
            valid_pred_dict[model_name],
            test_pred_dict[model_name],
        )


In [ ]:
from src.evaluation import Scorer

# Score only predictions produced by this experiment.
scorer_train = Scorer(train_pred_dict)
print("Training Data:")
scorer_train.show_metrics(truncate = 80)

scorer_valid = Scorer(valid_pred_dict)
print("Validation Data:")
scorer_valid.show_metrics(truncate = 80)

scorer_test = Scorer(test_pred_dict)
print("Testing Data:")
scorer_test.show_metrics(truncate = 80)